# PosturTakip — Fine-Tuning Notebook

**Bu notebook Llama 3.2 3B modelini postur analizi için fine-tune eder.**

### Başlamadan önce:
- Üst menü: **Runtime > Change runtime type > T4 GPU** seç
- `training_data.jsonl` dosyasını Google Drive ana klasörüne yükle

### Adımlar:
1. Paketleri kur → Runtime yeniden başlat
2. GPU kontrolü
3. Model yükle (Llama 3.2 3B)
4. LoRA adaptörleri ekle
5. Veriyi yükle
6. Fine-tune et (10-20 dk)
7. GGUF'a dönüştür → Drive'a kaydet

In [ ]:
# ADIM 1: Paketleri kur
# Bu hücreyi çalıştırdıktan sonra: Runtime > Restart session
%%capture
!pip install unsloth trl datasets transformers accelerate bitsandbytes peft

In [ ]:
# ADIM 2: GPU kontrolü
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU bulunamadi! Runtime > Change runtime type > T4 GPU sec!")

gpu_name = torch.cuda.get_device_name(0)
gpu_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU  : {gpu_name}")
print(f"VRAM : {gpu_vram:.1f} GB")
print("Devam edebiliriz!")

In [ ]:
# ADIM 3: Llama 3.2 3B modelini yükle (4-bit quantization ile VRAM tasarrufu)
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
print("Model yuklendi!")

In [ ]:
# ADIM 4: LoRA adaptörleri ekle (eğitilecek parametreler)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("LoRA adaptörleri eklendi!")
model.print_trainable_parameters()

In [ ]:
# ADIM 5: Google Drive'ı bağla ve training_data.jsonl'yi yükle
from google.colab import drive
drive.mount('/content/drive')

import json

DATA_PATH = '/content/drive/MyDrive/training_data.jsonl'

training_data = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            training_data.append(json.loads(line))

print(f"{len(training_data)} egitim ornegi yuklendi.")

# Veriyi modelin formatına dönüştür
from datasets import Dataset

def format_chat(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_chat, remove_columns=dataset.column_names)

print(f"Dataset hazir: {len(dataset)} ornek")
print("\nOrnek format (ilk 300 karakter):")
print(dataset[0]['text'][:300])

In [ ]:
# ADIM 6: Fine-tuning egitimi (10-20 dakika sürer)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=5,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

print("Egitim basliyor...")
stats = trainer.train()
print(f"\nEgitim tamamlandi!")
print(f"  Son kayip : {stats.training_loss:.4f}")
print(f"  Sure      : {stats.metrics['train_runtime']:.0f} saniye")

In [ ]:
# ADIM 7: GGUF formatına dönüştür ve Google Drive'a kaydet
import os, shutil

print("GGUF formatina donusturuluyor... (5-10 dakika)")
model.save_pretrained_gguf(
    "posturtakip-model",
    tokenizer,
    quantization_method="q4_k_m"
)

# Oluşturulan GGUF dosyalarını bul ve Drive'a kopyala
gguf_files = [f for f in os.listdir('.') if f.endswith('.gguf')]
print(f"\nOlusturulan dosyalar: {gguf_files}")

for fname in gguf_files:
    dest = f"/content/drive/MyDrive/{fname}"
    shutil.copy(fname, dest)
    size_mb = os.path.getsize(fname) / 1024**2
    print(f"  {fname} ({size_mb:.0f} MB) -> Google Drive'a kaydedildi")

print("\nFine-tuning tamamlandi!")
print("Sonraki adim: GGUF dosyasini bilgisayara indir")
print("Ardından: .\\backend\\scripts\\ollama-load-finetuned.ps1 calistir")

## Fine-Tuning Tamamlandı!

### Sonraki adımlar (bilgisayarında):

1. Google Drive'dan `posturtakip-model-unsloth.Q4_K_M.gguf` dosyasını indir

2. PowerShell'de şunu çalıştır:
```powershell
cd C:\Users\...\Desktop\postur-takip
.\backend\scripts\ollama-load-finetuned.ps1 -GGUFPath 'C:\Users\...\Downloads\posturtakip-model-unsloth.Q4_K_M.gguf'
```

3. Backend'i yeniden başlat:
```powershell
docker-compose restart backend
```

Artık uygulama fine-tuned modeli kullanacak!